# FedASIO-YOLO26 Comparative Study Notebook 🧠

This notebook enables you to run the comparative study of YOLO26 + ASIO against YOLO12 configurations in a federated learning framework using the balanced 20-patient BraTS dataset.

In [ ]:
# Verify environment and imports
import os
import sys
import torch
import ultralytics
import flwr
import nibabel

print("Versions:")
print(f"  PyTorch:     {torch.__version__}")
print(f"  CUDA/GPU:    {torch.cuda.is_available()} (Device: {'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')})")
print(f"  Ultralytics: {ultralytics.__version__}")
print(f"  Flower:      {flwr.__version__}")
print(f"  Nibabel:     {nibabel.__version__}")

## 1. Setup Dataset Path & Validation
Check if the dataset exists under `/Users/shrikant/Downloads/BraTS-PEDs-v1/Training`.

In [ ]:
RAW_DIR = "/Users/shrikant/Downloads/BraTS-PEDs-v1/Training"
if os.path.exists(RAW_DIR):
    patients = [d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d)) and d.startswith("BraTS")]
    print(f"✅ Dataset found! Total patients: {len(patients)}")
    print(f"Sample patients: {patients[:5]}")
else:
    print(f"❌ Dataset not found at: {RAW_DIR}")
    print("Please make sure the dataset folder is correctly placed or symlinked.")

## 2. Model Weight Check
Verify that the base pre-trained models exist locally in `yolo vs 26`.

In [ ]:
yolo12_path = "/Users/shrikant/Downloads/yolo vs 26/yolo12n-seg.pt"
yolo26_path = "/Users/shrikant/Downloads/yolo vs 26/yolo26n-seg.pt"

print(f"YOLO12 weights exist: {os.path.exists(yolo12_path)}")
print(f"YOLO26 weights exist: {os.path.exists(yolo26_path)}")

## 3. Run the Comparative Study (Dry Run)
We will first run a short Dry-Run (2 Rounds, 1 Local Epoch) to verify the end-to-end flow and confirm we get non-zero Dice scores.

In [ ]:
# Run the dry run
!python scripts/run_comparative_study.py --dry-run

## 4. Run the Full Comparative Study
Run the full comparative study benchmark (5 Rounds, 5 Local Epochs) to train the federated models.

In [ ]:
# Run the full comparative study (this will take longer)
!python scripts/run_comparative_study.py --rounds 5 --epochs 5

## 5. Visualize Results
Plot and inspect the metrics generated by the comparative study.

In [ ]:
import json

metrics_file = "/Users/shrikant/Downloads/FedASIO-YOLO26/reports/metrics/comparison_study.json"
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        results = json.load(f)
    print("Comparison Study Results:")
    print(json.dumps(results, indent=2))
    
    # Show the bar chart
    fig_path = "/Users/shrikant/Downloads/FedASIO-YOLO26/reports/figures/comparison_study.png"
    if os.path.exists(fig_path):
        from IPython.display import Image, display
        display(Image(filename=fig_path))
else:
    print("Metrics file not found. Please run the study first.")